In [ ]:
# Python ≥3.9 is required
import sys
assert sys.version_info >= (3, 9)

import seaborn as sns

# Statistical analysis and testing
from scipy import stats
from statsmodels.formula.api import ols
import statsmodels.api as sm

# Ignore useless warnings (see SciPy issue #5998)
import warnings
warnings.filterwarnings(action="ignore", message="^internal gelsd")

In [ ]:
# load in the  dataset
nhanes = pd.read_csv(
    "nhanes.csv"
)

In [ ]:
nhanes.info()

In [ ]:
nhanes.sample(5)

In [ ]:
nhanes.describe()

In [ ]:
counts_diabetes = nhanes['DIQ010'].value_counts()
counts_diabetes

In [ ]:
count_over_65_hba1c = (nhanes['LBXGH'] >= 6.5).sum()
count_over_65_hba1c

In [ ]:
count_over_21_age = (nhanes['RIDAGEYR'] >= 21).sum()
count_over_21_age

In [ ]:
#drop if under 21
nhanes = nhanes[nhanes['RIDAGEYR'] >= 21]

In [ ]:
counts_diabetes = nhanes['DIQ010'].value_counts()
counts_diabetes

In [ ]:
count_over_65_hba1c = (nhanes['LBXGH'] >= 6.5).sum()
count_over_65_hba1c

In [ ]:
# sensitivity test
# exclusion of med conditions - 3k and ~84 diabetics, 49 borderline, and 75 had an hba1c level above 6.5 therefore will not drop
""""
nhanes = nhanes[~nhanes['MCQ510A'].isin([1, 77, 99])]
nhanes = nhanes[nhanes['MCQ510B'] != 2]
nhanes = nhanes[nhanes['MCQ510C'] != 3]
nhanes = nhanes[nhanes['MCQ510D'] != 4]
nhanes = nhanes[nhanes['MCQ510E'] != 5]
nhanes = nhanes[nhanes['MCQ510F'] != 6]

vars_to_filter = [
    'BPQ020','BPQ150','BPQ101D','MCQ160B','MCQ160C','MCQ160D',
    'MCQ160E','MCQ160F','MCQ160M','MCQ170M','MCQ160L','MCQ170L', 'MCQ220'
]

# remove rows/patients that have 1, 7, or 9
nhanes = nhanes[~nhanes[vars_to_filter].isin([1, 7, 9]).any(axis=1)]
""""

In [ ]:
""""
cols_to_drop = [
    'BPQ020','BPQ150','BPQ101D','MCQ160B','MCQ160C','MCQ160D',
    'MCQ160E','MCQ160F','MCQ160M','MCQ170M','MCQ160L','MCQ170L', 'MCQ220',
    'MCQ510A','MCQ510B','MCQ510C','MCQ510D','MCQ510E','MCQ510F'
]

# Drop the screening variables from nhanes
nhanes = nhanes.drop(columns=cols_to_drop)
""""

In [ ]:
edu_order = [1, 2, 3, 4, 5, 7, 9]

nhanes['DMDEDUC2'] = pd.Categorical(
    nhanes['DMDEDUC2'],
    categories=edu_order,
    ordered=True
)

# nhanes['DMDEDUC2'] = nhanes['DMDEDUC2'].replace({7: np.nan, 9: np.nan})

In [ ]:
alq_order = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 77, 99]

nhanes['ALQ121'] = pd.Categorical(
    nhanes['ALQ121'],
    categories=alq_order,
    ordered=True
)

# nhanes['ALQ121'] = nhanes['ALQ121'].replace({77: np.nan, 99: np.nan})

In [ ]:
diq_order = [1, 2, 3, 7, 9]

nhanes['DIQ010'] = pd.Categorical(
    nhanes['DIQ010'],
    categories=diq_order,
    ordered=True
)

# nhanes['DIQ010'] = nhanes['DIQ010'].replace({7: np.nan, 9: np.nan})

In [ ]:
fs_order = [1, 2, 3, 4]

nhanes['FSDAD'] = pd.Categorical(
    nhanes['FSDAD'],
    categories=fs_order,
    ordered=True
)

In [ ]:
huq_order = [1, 2, 3, 4, 5, 7, 9]

nhanes['HUQ010'] = pd.Categorical(
    nhanes['HUQ010'],
    categories=huq_order,
    ordered=True
)

# nhanes['HUQ010'] = nhanes['HUQ010'].replace({7: np.nan, 9: np.nan})

In [ ]:
smq_order = [1, 2, 3, 7, 9]

nhanes['SMQ040'] = pd.Categorical(
    nhanes['SMQ040'],
    categories=smq_order,
    ordered=True
)

# nhanes['SMQ040'] = nhanes['SMQ040'].replace({7: np.nan, 9: np.nan})

In [ ]:
from sklearn.model_selection import train_test_split
train_set, test_set = train_test_split(nhanes, test_size=0.2, random_state=13)

In [ ]:

variables_histoplot_eda = [
    "RIDAGEYR","RIDRETH3","DMDBORN4","DMDEDUC2","INDFMPIR",
    "BMXBMI","BMXWAIST","BMXWT","URDACT","LBDHDD","LBXTLG","LBDLDLM","LBXTC","LBXGH","LBXIN",
    "LBXGLU","ALQ121","ALQ130","DIQ010","DID040","DIQ050","DID060",
    "DIQ060U","DIQ070","FSDAD","HUQ010","SMQ040","PAD790Q","PAD810Q","BPQ020","BPQ150","BPQ101D",
    "MCQ160B","MCQ160C","MCQ160D","MCQ160E","MCQ160F","MCQ160M","MCQ170M","MCQ160L","MCQ170L",
    "MCQ510A","MCQ510B","MCQ510C","MCQ510D","MCQ510E","MCQ510F","MCQ220","LBXFER",
    "FNQ510","FNQ520","FNQ530","FNQ540","KIQ022","SLD012","SLD013"
]

n_vars = len(variables_histoplot_eda)
n_cols = 4
n_rows = (n_vars + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, n_rows*4))
axes = axes.flatten()

for i, var in enumerate(variables_histoplot_eda):
    if var in nhanes.columns:
        nhanes[var].hist(bins=30, ax=axes[i], color='skyblue', edgecolor='black')
        axes[i].set_title(var)
    else:
        axes[i].set_visible(False)


for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# create a training and test set - 20% of the dataset was set away for the test set
from sklearn.model_selection import train_test_split
nhanes = nhanes.dropna(subset=["LBXGH"])
X = nhanes.drop(columns=["LBXGH"])
y = (nhanes["LBXGH"] >= 6.5).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    stratify=y,
    test_size=0.2,
    random_state=42
)

In [ ]:
X_train.shape, X_test.shape

In [ ]:
# visualize data to gain insights - outlier detection
# values outside the IQR or the whiskers of the box plot can be interpreted as outliers
p = X_train[
    ['RIAGENDR','RIDAGEYR','RIDRETH3',
    'DMDEDUC2','INDFMPIR','BMXBMI','BMXWAIST','BMXWT','URDACT',
    'LBDHDD','LBXTLG','LBDLDLM','LBXTC','LBXIN','LBXGLU',
    'ALQ121','ALQ130','FSDAD','HUQ010','SMQ040','PAD790Q',
    'PAD810Q','LBXFER','FNQ510','FNQ520','FNQ530',
    'FNQ540','KIQ022','SLD012','SLD013']
].plot.box(subplots=True, layout=(8,4), figsize=(18,18))

In [ ]:
# number of outliers were counted and removed for the outcome variable, LBXGH
q1 = train_set['LBXGH'].quantile(0.25)
q3 = train_set['LBXGH'].quantile(0.75)
IQR = q3 - q1

outlier_lowerbound = q1 - 1.5 * IQR
outlier_upperbound = q3 + 1.5 * IQR

outliers_select = train_set[(train_set['LBXGH'] < outlier_lowerbound) | \
    (train_set['LBXGH'] > outlier_upperbound)]
outliers_count = len(outliers_select)

print(f"Count of outliers with respect to LBXGH': {outliers_count}")

train_set = train_set[(train_set['LBXGH'] >= outlier_lowerbound) & \
    (train_set['LBXGH'] <= outlier_upperbound)]


In [ ]:
train_set.shape

In [ ]:
# Identify correlations by computing correlation coefficients (Pearsons's r coefficient) by every 2 features
# This can be done as the dataset is not big
train_df = X_train.copy()
train_df["LBXGH"] = y_train  # add target back temporarily

corr_matrix = train_df.corr(numeric_only=True)
corr_matrix["LBXGH"].sort_values(ascending=False)

In [ ]:
col_cont = [
    'RIDAGEYR', 'INDFMPIR','BMXBMI','BMXWAIST','BMXWT','URDACT',
    'LBDHDD','LBXTLG','LBDLDLM','LBXGH', 'LBXTC','LBXIN','LBXGLU',
    'ALQ130','PAD790Q',
    'PAD810Q','LBXFER','FNQ510','FNQ520','FNQ530',
    'FNQ540','KIQ022','SLD012','SLD013'
]



In [ ]:
# use scatter_matrix to see if there are non-linear correlation factors
pd.plotting.scatter_matrix(
    train_df[col_cont], figsize=(15, 10)
)
plt.show()

In [ ]:
ax = sns.pairplot(train_df[col_cont])

In [ ]:
# Checking multicolinearity through using a seaborn heatmap
# remove sequence number
train_df_pred = train_df.drop(["SEQN","SDDSRVYR","RIDSTATR"], axis=1)

# Compute the correlation matrix
corr = round(train_df_pred.corr(numeric_only=True), 3)

# Set up the matplotlib figure
fig, ax = plt.subplots(figsize=(15, 12))

# Generate a diverging colormap
cmap = sns.diverging_palette(230, 20, as_cmap=True)

# Draw the heatmap
ax = sns.heatmap(corr, cmap=cmap, vmax=1, vmin=-1, center=0, square=True, linewidths=1, \
    cbar_kws={"shrink": .75}, annot=True)

plt.show()

In [ ]:
# Generate a triangular boolean mask to get rid of the upper triangle, since it is symmetrical/identical
mask = np.triu(np.ones_like(corr, dtype=bool))

# Use matplotlib to visualize the mask
fig, ax = plt.subplots()

ax.matshow(mask, cmap="gray")

for (i, j), z in np.ndenumerate(mask):
    ax.text(j, i, '{:0.0f}'.format(z), ha='center', va='center', c="gray")

plt.show()

In [ ]:
# set up the matplotlib figure
fig, ax = plt.subplots(figsize=(15, 12))

# Draw the heatmap with the mask
ax = sns.heatmap(corr, mask=mask, cmap=cmap, vmax=1, vmin=-1, center=0, square=True, linewidths=1, cbar_kws={"shrink": .75}, annot=True)

plt.show()

In [ ]:
#assessing numerical data for linear relationships with outcome


for feat in col_cont:
    sns.jointplot(
        x=feat,
        y="LBXGH",
        data=train_df,
        kind='reg',
        label=feat,
        joint_kws={'line_kws':{'color':'orange'}}
    )
    plt.legend()
    plt.show()

In [ ]:
col_cat = [
    'RIAGENDR','RIDRETH3',
    'DMDEDUC2','ALQ121','FSDAD','HUQ010','SMQ040'
]

In [ ]:
# assess categorical features
# first ocean proximity, which is the categorical feature, has to be converted into a category type
train_df[col_cat] = train_df[col_cat].astype('category')
train_df.info()

In [ ]:
# boxplot was used as a categorical variable is being plotted against a numerical one which housing price
# the boxplot was created to plot price distribution against each category of ocean proximity
def print_boxplot(
    data_set: pd.DataFrame,
    feature: str,
    figsize: tuple[int, int] = (14, 6),
    xlabels_rotation: int = None
):

    # create a dataframe containing only feature and "price"
    data = data_set[[feature, "LBXGH"]].copy()
    # size of figure
    plt.figure(figsize=figsize)
    # seaborn to create the boxplot
    chart = sns.boxplot(x=feature, y="LBXGH", data=data)
    # rotate the x label
    if xlabels_rotation is not None:
        chart.set_xticklabels(chart.get_xticklabels(), rotation=xlabels_rotation, horizontalalignment='center')

In [ ]:
print_boxplot(train_df, 'RIAGENDR')

In [ ]:
print_boxplot(train_df, 'RIDRETH3')

In [ ]:
print_boxplot(train_df, 'DMDEDUC2')

In [ ]:
print_boxplot(train_df, 'ALQ121')

In [ ]:
print_boxplot(train_df, 'FSDAD')

In [ ]:
print_boxplot(train_df, 'HUQ010')

In [ ]:
print_boxplot(train_df, 'SMQ040')

In [ ]:
# check for missing values
incomplete_rows = X_train[X_train.isna().any(axis=1)]
incomplete_rows

In [ ]:
# Feature engineering based on the conclusions from the EDA
def engineer_features(data: pd.DataFrame) -> pd.DataFrame:
    # copy of the original dataset
    engineered_data = data.copy()
    # replace 0 rooms with NaN (we will fill it later on)
    engineered_data.loc[engineered_data['BMXBMI'] == 0, 'BMXBMI'] = np.nan
    # replace 0 rooms with NaN (we will fill it later on)
    engineered_data.loc[engineered_data['LBXGLU'] == 0, 'LBXGLU'] = np.nan
    # replace 0 rooms with NaN (we will fill it later on)
    engineered_data.loc[engineered_data['LBXTC'] == 0, 'LBXTC'] = np.nan
    # replace 0 rooms with NaN (we will fill it later on)
    engineered_data.loc[engineered_data['LBXIN'] == 0, 'LBXIN'] = np.nan
    # replace 0 rooms with NaN (we will fill it later on)
    engineered_data.loc[engineered_data['LBDHDD'] == 0, 'LBDHDD'] = np.nan
    # replace 0 rooms with NaN (we will fill it later on)
    engineered_data.loc[engineered_data['LBXFER'] == 0, 'LBXFER'] = np.nan
    # replace 0 rooms with NaN (we will fill it later on)
    engineered_data.loc[engineered_data['RIAGENDR'] == 0, 'RIAGENDR'] = np.nan
    # replace 0 rooms with NaN (we will fill it later on)
    engineered_data.loc[engineered_data['RIDRETH3'] == 0, 'RIDRETH3'] = np.nan
    # replace 0 rooms with NaN (we will fill it later on)
    engineered_data.loc[engineered_data['DMDEDUC2'] == 0, 'DMDEDUC2'] = np.nan
    # replace 0 rooms with NaN (we will fill it later on)
    engineered_data.loc[engineered_data['FSDAD'] == 0, 'FSDAD'] = np.nan
    # replace 0 rooms with NaN (we will fill it later on)
    engineered_data.loc[engineered_data['HUQ010'] == 0, 'HUQ010'] = np.nan
    # replace 0 rooms with NaN (we will fill it later on)
    engineered_data.loc[engineered_data['SMQ040'] == 0, 'SMQ040'] = np.nan
    # drop
    engineered_data = engineered_data.drop(columns=["BMXWAIST"])
    # drop
    engineered_data = engineered_data.drop(columns=["BMXWT"])
    # drop
    engineered_data = engineered_data.drop(columns=["LBXTC"])
    # drop id, cycle, completed interview flag (as we won't use them)
    engineered_data = engineered_data.drop(columns=["SEQN", "SDDSRVYR","RIDSTATR"])
    return engineered_data

In [ ]:
train_data = engineer_features(X_train)
train_labels = y_train.copy()
train_data.sample(10, random_state=42)

In [ ]:
len(train_data.columns)

In [ ]:
# identify missing data for imputation (SimpleImputer would replace
# the numeric values with the median value for that feature)
incomplete_rows = train_data[train_data.isna().any(axis=1)]
incomplete_rows

In [ ]:
# Create custom tranformer
from sklearn.base import BaseEstimator, TransformerMixin

class FeatureEngineeringTransformer(BaseEstimator, TransformerMixin):
    def fit(self, data, labels=None) -> "FeatureEngineeringTransformer":
        return self

    def transform(self, data, labels=None) -> pd.DataFrame:
        # copy of the original dataset
        engineered_data = data.copy()
        # replace 0 rooms with NaN (we will fill it later on)
        engineered_data.loc[engineered_data['BMXBMI'] == 0, 'BMXBMI'] = np.nan
        # replace 0 rooms with NaN (we will fill it later on)
        engineered_data.loc[engineered_data['LBXGLU'] == 0, 'LBXGLU'] = np.nan
        # replace 0 rooms with NaN (we will fill it later on)
        engineered_data.loc[engineered_data['LBXTC'] == 0, 'LBXTC'] = np.nan
        # replace 0 rooms with NaN (we will fill it later on)
        engineered_data.loc[engineered_data['LBXIN'] == 0, 'LBXIN'] = np.nan
        # replace 0 rooms with NaN (we will fill it later on)
        engineered_data.loc[engineered_data['LBDHDD'] == 0, 'LBDHDD'] = np.nan
        # replace 0 rooms with NaN (we will fill it later on)
        engineered_data.loc[engineered_data['LBXFER'] == 0, 'LBXFER'] = np.nan
        # replace 0 rooms with NaN (we will fill it later on)
        engineered_data.loc[engineered_data['RIAGENDR'] == 0, 'RIAGENDR'] = np.nan
        # replace 0 rooms with NaN (we will fill it later on)
        engineered_data.loc[engineered_data['RIDRETH3'] == 0, 'RIDRETH3'] = np.nan
        # replace 0 rooms with NaN (we will fill it later on)
        engineered_data.loc[engineered_data['DMDEDUC2'] == 0, 'DMDEDUC2'] = np.nan
        # replace 0 rooms with NaN (we will fill it later on)
        engineered_data.loc[engineered_data['FSDAD'] == 0, 'FSDAD'] = np.nan
        # replace 0 rooms with NaN (we will fill it later on)
        engineered_data.loc[engineered_data['HUQ010'] == 0, 'HUQ010'] = np.nan
        # replace 0 rooms with NaN (we will fill it later on)
        engineered_data.loc[engineered_data['SMQ040'] == 0, 'SMQ040'] = np.nan
        # drop
        engineered_data = engineered_data.drop(columns=["BMXWAIST"])
        # drop
        engineered_data = engineered_data.drop(columns=["BMXWT"])
        # drop
        engineered_data = engineered_data.drop(columns=["LBXTC"])
        # drop id, cycle, completed interview flag (as we won't use them)
        engineered_data = engineered_data.drop(columns=["SEQN", "SDDSRVYR","RIDSTATR"])
        return engineered_data


In [ ]:
#check to see custom transformer works
fe_trf = FeatureEngineeringTransformer()
fe_trf.fit(
    X_train,
    y_train
)
fe_trf.transform(X_train)

,RIAGENDR,RIDAGEYR,RIDRETH3,DMDEDUC2,INDFMPIR,BMXBMI,URDACT,LBDHDD,LBXTLG,LBDLDLM,...,PAD790Q,PAD810Q,LBXFER,FNQ510,FNQ520,FNQ530,FNQ540,KIQ022,SLD012,SLD013
7256,1.00,68,3.00,3.00,1.26,26.50,110.83,49.00,190.00,137.00,...,5.00,0.00,NaN,4.00,3.00,4.00,1.00,2.00,8.50,8.50
7972,2.00,27,3.00,5.00,5.00,18.50,0.55,63.00,55.00,56.00,...,2.00,3.00,35.30,2.00,2.00,4.00,3.00,2.00,8.00,10.00
113,2.00,69,7.00,4.00,0.79,36.60,53.08,48.00,NaN,NaN,...,2.00,2.00,NaN,2.00,3.00,1.00,2.00,2.00,6.00,6.00
1535,1.00,80,4.00,3.00,NaN,32.40,118.80,64.00,36.00,86.00,...,1.00,0.00,NaN,5.00,NaN,5.00,NaN,2.00,10.00,10.00
2319,1.00,12,1.00,NaN,3.40,26.50,0.91,54.00,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6648,2.00,36,3.00,4.00,2.09,31.30,6.26,45.00,97.00,112.00,...,2.00,0.00,79.00,2.00,1.00,4.00,1.00,2.00,8.50,10.00
6080,1.00,73,3.00,4.00,1.99,35.20,60.94,44.00,NaN,NaN,...,2.00,0.00,NaN,5.00,NaN,5.00,NaN,1.00,10.00,10.00
3764,1.00,20,3.00,4.00,1.65,21.30,6.70,48.00,NaN,NaN,...,20.00,0.00,NaN,3.00,3.00,5.00,NaN,2.00,5.50,9.00
676,1.00,56,4.00,2.00,0.75,21.60,221.97,36.00,NaN,NaN,...,0.00,0.00,NaN,5.00,NaN,5.00,NaN,2.00,3.50,NaN


In [ ]:
# Feature scaling and transformation pipeline (chain transformation from StandardScaler and SimpleImputer)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

num_feats = [
    el for el in list(
        train_data.select_dtypes(include=[np.number])
    )
]

In [ ]:
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy="median")),
    ('std_scaler', StandardScaler())
])
train_data_num_scaled = num_pipeline.fit_transform(train_data[num_feats])

In [ ]:
train_data_num_scaled[:5]

In [ ]:
# create Pandas output instead of NumPy arrays
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy="median").set_output(transform="pandas")),
    ('std_scaler', StandardScaler().set_output(transform="pandas"))
])
train_data_num_scaled = num_pipeline.fit_transform(train_data[num_feats])

In [ ]:
train_data_num_scaled.head()

In [ ]:
from sklearn.preprocessing import OneHotEncoder
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy="most_frequent")),
    ('onehot', OneHotEncoder(handle_unknown="ignore", sparse_output=False).set_output(transform="pandas"))
])

In [ ]:
#chaining ColumnsTransformer and Pipeline
from sklearn.compose import ColumnTransformer

column_transformer = ColumnTransformer(
    (
        ("numerical", num_pipeline, num_feats),
        ("categorical", cat_pipeline, col_cat),
    ),
    remainder="passthrough",
    verbose_feature_names_out=False,
).set_output(transform="pandas")

column_transformer

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","(('numerical', ...), ...)"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.

In [ ]:
train_data_prepared = column_transformer.fit_transform(train_data)
train_data_prepared

In [ ]:
from sklearn.pipeline import make_pipeline

full_pipeline = make_pipeline(FeatureEngineeringTransformer(), column_transformer)
full_pipeline

In [ ]:
print(X_train.columns)
full_pipeline.fit(X_train, y_train)

In [ ]:
train_data_prepared =  full_pipeline.transform(X_train)
train_data_prepared

In [ ]:
# Prepare test set - only transform was used and not fit
test_data_prepared = full_pipeline.transform(X_test)

In [ ]:
test_data_prepared.head()

In [ ]:
train_export = train_data_prepared.copy()
train_export["target"] = y_train

train_export.to_csv("/train_bin_nhanes.csv", index=False)

In [ ]:
test_export = test_data_prepared.copy()
test_export["target"] = y_test

test_export.to_csv("/test_bin_nhanes.csv", index=False)

In [ ]:
train_data_prepared.info()